# 01 · Teammate qualifying gaps

Teammates share (nearly) the same car, so their qualifying gap is the cleanest early signal of driver pace.

**Method:** for each race, the two teammates are compared in the last qualifying segment they *both* set a time in (Q3 if both reached it, else Q2, else Q1). Gap is in % of the teammate's lap: negative = faster. Gaps above the outlier threshold are flagged and excluded from season summaries.

**Known limitation:** teammates' cars are not identical (setups, upgrade timing, reliability), so this is an approximation.

Run `00_ingest_quali.ipynb` first.

In [ ]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/YOUR_USERNAME/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

In [ ]:
# @title Load data and compute gaps
OUTLIER_PCT = 2.0  # @param {type:"number"}

import plotly.express as px
from f1sim.paths import get_paths
from f1sim.ingest import load_quali
from f1sim.features import teammate_gaps, season_summary

quali = load_quali(get_paths())
gaps = teammate_gaps(quali, outlier_pct=OUTLIER_PCT)
summary = season_summary(gaps)

n_sessions = len(quali.drop_duplicates(["Year", "Round"]))
print(f"{n_sessions} sessions | {len(gaps) // 2} teammate comparisons | "
      f"{int(gaps['Outlier'].sum()) // 2} flagged as outliers")

In [ ]:
# @title Season head-to-head table
YEAR = 2025  # @param {type:"integer"}

summary.loc[summary["Year"] == YEAR, ["Team", "Driver", "Teammate", "Races", "H2H", "MedianGapPct"]]

In [ ]:
# @title Median gap to teammate, by driver
season = summary[summary["Year"] == YEAR]
fig = px.bar(
    season.sort_values("MedianGapPct"),
    x="MedianGapPct", y="Driver", color="Team", orientation="h",
    hover_data=["Teammate", "H2H", "Races"],
    labels={"MedianGapPct": "Median gap to teammate (%)"},
    title=f"{YEAR}: median qualifying gap to teammate (negative = faster)",
)
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(height=650, yaxis={"categoryorder": "total descending"})
fig.show()

In [ ]:
# @title One team, race by race
TEAM = "McLaren"  # @param {type:"string"}

teams = sorted(gaps.loc[gaps["Year"] == YEAR, "Team"].unique())
if TEAM not in teams:
    print(f"'{TEAM}' not found in {YEAR}. Pick one of:", teams)
else:
    tg = gaps[(gaps["Year"] == YEAR) & (gaps["Team"] == TEAM)]
    fig = px.line(
        tg, x="Round", y="GapPct", color="Driver", markers=True, symbol="Outlier",
        hover_data=["Event", "Teammate", "Segment"],
        labels={"GapPct": "Gap to teammate (%)"},
        title=f"{TEAM} {YEAR}: qualifying gap by round",
    )
    fig.add_hline(y=0, line_dash="dot")
    fig.show()

In [ ]:
# @title One driver across seasons
DRIVER = "ALO"  # @param {type:"string"}

d = summary[summary["Driver"] == DRIVER].copy()
if d.empty:
    print(f"No data for '{DRIVER}'. Use the 3-letter code, e.g. HAM, VER, LEC.")
else:
    fig = px.line(
        d, x="Year", y="MedianGapPct", markers=True, text="Teammate",
        hover_data=["Team", "H2H", "Races"],
        labels={"MedianGapPct": "Median gap to teammate (%)"},
        title=f"{DRIVER}: median qualifying gap to teammate by season",
    )
    fig.update_traces(textposition="top center")
    fig.add_hline(y=0, line_dash="dot")
    fig.show()

In [ ]:
# @title Inspect flagged outliers for the season
gaps.loc[gaps["Outlier"] & (gaps["Year"] == YEAR), ["Round", "Event", "Team", "Driver", "Teammate", "Segment", "GapPct"]]